In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go #type ignore
import plotly.express as px
import plotly.io as pio

In [2]:
# Data import
spy = pd.read_csv('/Users/Ben.Pharris/Downloads/7UMUD0MZ.csv')

# --- Creating Timestamped Date
hour_times = ["09:30", "10:30", "11:30", "12:30", "13:30", "14:30", "15:30"]

# Create a counter 0–6 within each date
spy['hour_index'] = spy.groupby('Date').cumcount()

# Overwrite Date with timestamp
spy['Date'] = spy.apply(
    lambda row: pd.Timestamp(f"{row['Date']} {hour_times[row['hour_index']]}"),
    axis=1
)

# Drop helper column
spy.drop(columns='hour_index', inplace=True)
del(hour_times) 

In [3]:
# buy and hold return
r_bh = spy['Close'].iloc[-1] - spy['Close'].iloc[0]
print(r_bh)

# buy and hold % return
r_bh_pct = spy['Close'].iloc[-1] / spy['Close'].iloc[0] - 1
print(r_bh_pct)




106.40997319999997
0.1909760992509868


In [4]:
# intraday returns
spy['r_h'] = spy['Close'] - spy['Open']
# intraday returns, buy or hold cash
spy['buy_if_up'] = spy['r_h'].where(spy['r_h'] > 0, 0)
# perfect information, short+long hourly return
spy['ppsl'] = abs(spy['Close'] - spy['Open'])


fig = go.Figure(go.Waterfall(
    x = spy['Date'],
    y = spy['r_h'],
    name= "Intraday Return",
))

fig.add_trace(go.Waterfall(x = spy['Date'], y = spy['buy_if_up'], name = "Intraday Return - Gains Only"))

fig.add_trace(go.Waterfall(x = spy['Date'], y = spy['ppsl'], name = "Intraday Return -  Pure Profit"))

fig.update_layout(title="Intraday Returns - Waterfall Chart")
fig.show()

In [5]:
spy['r_h'].describe()



count    1871.000000
mean        0.019840
std         1.821937
min       -11.859985
25%        -0.679993
50%         0.050049
75%         0.779968
max        24.770020
Name: r_h, dtype: float64

In [6]:
# create a binary label for positive/negative returns
spy['direction'] = spy['r_h'].apply(lambda x: 'Positive' if x > 0 else 'Negative')

# create histogram with visual separation
fig = px.histogram(
    spy,
    x="direction",
    color="direction",
    text_auto=True,             # show counts on bars
    category_orders={"direction": ["Negative", "Positive"]},
    color_discrete_map={'Positive': '#2ecc71', 'Negative': '#e74c3c'},  # green/red
)

# styling
fig.update_layout(
    title="Distribution of Hourly Returns (r_h)",
    xaxis_title="Return Direction",
    yaxis_title="Count",
    bargap=0.3,                     # separation between bars
    showlegend=False,
    plot_bgcolor="white",
)

fig.update_traces(textfont_size=14, textposition='outside')

fig.show()

In [7]:
fig = px.histogram(spy, x='r_h', nbins=40, range_x=[-10, 10], text_auto=True)
fig.update_layout(bargap = 0.2, title = "Distribution of Intraday Returns",
                   xaxis_title = "Hourly Returns", yaxis_title = "Count"  )

fig.show()

In [8]:
import plotly.io as pio

# intraday returns
spy["r_h"] = spy["Close"] - spy["Open"]
# intraday returns, buy or hold cash
spy["buy_if_up"] = spy["r_h"].where(spy["r_h"] > 0, 0)
# perfect information, short+long hourly return
spy["ppsl"] = abs(spy["Close"] - spy["Open"])

# build figure
fig = go.Figure()

fig.add_trace(go.Waterfall(
    x = spy["Date"],
    y = spy["r_h"],
    name = "Buy and Hold Return",
    connector = {"visible": False},
    increasing = {"marker": {"color": "#1f77b4"}},  # blue
    decreasing = {"marker": {"color": "#1f77b4"}}
))

fig.add_trace(go.Waterfall(
    x = spy["Date"],
    y = spy["buy_if_up"],
    name = "Upside Only - No Short",
    connector = {"visible": False},
    increasing = {"marker": {"color": "#2ca02c"}},  # green
    decreasing = {"marker": {"color": "#2ca02c"}}
))

fig.add_trace(go.Waterfall(
    x = spy["Date"],
    y = spy["ppsl"],
    name = "Pure Profit / Perfect Information",
    connector = {"visible": False},
    increasing = {"marker": {"color": "#ff7f0e"}},  # orange
    decreasing = {"marker": {"color": "#ff7f0e"}}
))

# clean layout
fig.update_layout(
    title="Intraday Returns",
    showlegend=True,
    width=800,
    height=800,  # square aspect ratio
    margin=dict(l=40, r=40, t=60, b=40),
    template="simple_white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

# (optional) display interactively
fig.show()
fig.write_image("IntradayReturns.png")